# 01_p1_version_policy — P1 버전 정책 (RF-06)

[RF-06 계획서](../../../docs/ml/VERSION_POLICY_EFFICIENCY_EXPERIMENT.md) 2절. 운영 `successfulHttpRequest` 규칙: 1.1·1.2는 M0(1.0)에서, 2.0은 B1~B3으로 M0에서, 2.1·2.2·3.0은 2.0에서 시작한다(가중치만, optimizer는 새로). 학습 행은 현재 cycle의 batch 전부 + 지난 cycle replay(라벨별 32) + D_seed 복습(라벨별 32).

- round마다 학습 → dev 채점 → 다음 probe window 채점(prequential) → 마지막 round는 test 채점(질문 원문만, 결과는 03에서 읽음).
- round마다 Drive `arms/<arm>/seed<N>/round<kk>/`에 receipt와 `_SUCCESS`를 쓴다. **끊기면 같은 셀을 다시 실행하면 끝난 round는 건너뛴다.**
- 모델 가중치는 다음 round에 필요한 것(P1의 x.0 anchor)과 마지막 round만 Drive에 남긴다.

**런타임: 2026.07(Python 3.12), T4 GPU.** 9 round 약 1.5~2.5시간(00 pilot의 step당 시간으로 다시 추정한다).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
RUN = Path('/content/drive/MyDrive/LLM_Router/version-policy/runs/run001')
SOURCE_ZIP = RUN / 'code' / 'version-policy-source.zip'
expected = SOURCE_ZIP.with_suffix('.sha256').read_text().split()[0]
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == expected, '소스 ZIP이 바뀌었다'
REPO = Path('/content/version-policy-source')
shutil.rmtree(REPO, ignore_errors=True)
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
for name, digest in json.loads((REPO / 'source-bundle-manifest.json').read_text()).items():
    assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == digest, name
assert sys.version_info[:2] in {(3, 11), (3, 12)}, '런타임 버전을 2026.07(Python 3.12)로 바꾸세요'

subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.5.1', 'torchvision==0.20.1', 'torchaudio==2.5.1',
                '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'ml/notebooks/version_policy/requirements-colab.txt')], check=True)

In [ ]:
CODE = REPO / 'ml/notebooks/version_policy'
DATA = RUN / 'data'
ENV = dict(os.environ, PYTHONPATH=str(REPO / 'ml/src'), PYTHONUNBUFFERED='1', TRANSFORMERS_VERBOSITY='error')

def run_logged(script, args):
    """Run a script in a fresh process (it sees the packages installed above) and stream its log here."""
    proc = subprocess.Popen([sys.executable, str(CODE / script), *map(str, args)], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=ENV, cwd=str(CODE))
    for line in proc.stdout:
        if 'Warning' not in line and 'warn(' not in line:
            print(line, end='')
    if proc.wait():
        raise RuntimeError(f'실패(exit {proc.returncode}). 위 로그의 마지막 부분(Traceback)을 보내 주세요')

assert (DATA / 'manifest.json').exists(), 'Drive에 data/(로컬 build 결과)를 먼저 올리세요. README 참고'
print(json.loads((DATA / 'manifest.json').read_text())['rows'])

import torch
assert torch.cuda.is_available(), 'T4 GPU 런타임을 선택하세요'
print(torch.cuda.get_device_name(0), torch.__version__)
assert (RUN / 'plan' / '_SUCCESS').exists(), '00_pilot의 봉인을 먼저 끝내세요'

## 1. 이 세션의 arm·seed

In [ ]:
ARM = 'p1'
SEED = 42   # 판정이 경계(±.01)에 걸리면 43·44를 추가한다(계획서 6절)

## 2. round 실행

In [ ]:
run_logged('rounds.py', ['--repo', REPO, '--data-dir', DATA, '--run-dir', RUN, '--arm', ARM, '--seed', SEED, '--device', 'cuda'])

## 3. 기록 확인

In [ ]:
for folder in sorted((RUN / 'arms' / ARM / f'seed{SEED}').glob('round*')):
    if (folder / '_SUCCESS').exists():
        r = json.loads((folder / 'receipt.json').read_text())
        print(f"round {r['round']} ({r['version']}): rows {r['train_rows']:>6} {r['rows_by_part']}, steps {r['optimizer_steps']:>5}, "
              f"fit {r['fit_seconds']/60:6.1f}분, score {r['score_seconds']/60:4.1f}분")